# Orion Config-Only Experiments

This notebook is intentionally thin.
All experiment knobs live in `configs/experiments/profiles/*.yaml` and `configs/experiments/variants/*.yaml`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/akashkguw/orion.git"
REPO_ROOT = Path("/content/orion") if IN_COLAB else Path.cwd()

if IN_COLAB and not REPO_ROOT.exists():
    print(f"Cloning {REPO_URL} -> {REPO_ROOT}")
    subprocess.run(["git", "clone", REPO_URL, str(REPO_ROOT)], check=True)

os.chdir(REPO_ROOT)
print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_ROOT={REPO_ROOT}")

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "-r",
        "requirements.txt",
        "-r",
        "requirements-dev.txt",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas", "seaborn"], check=True)

print("Dependencies installed.")

In [ ]:
from pathlib import Path

import seaborn as sns

from orion.experiments import (
    load_profile_context,
    load_summary_df,
    paired_analysis_tables,
    plot_all_numeric_metrics,
    plot_speedup_ratios,
    plot_vram_and_quality,
    prepare_analysis_columns,
    run_profile,
    select_winners,
)

sns.set_theme(style="whitegrid")

PROFILE = "pilot9"  # pilot9 | pilot | full | pilot_norm | w64_d_sweep
PROFILE_CTX = load_profile_context(PROFILE)
PROFILE_CTX

In [ ]:
run_result = run_profile(PROFILE)
run_result

In [ ]:
df = load_summary_df(run_result.summary_csv)
df_ok = prepare_analysis_columns(df[df["status"] == "ok"].copy())

print("Successful trials:", len(df_ok), "/", len(df))
display(df_ok.head(20))

In [ ]:
paired_all, paired, agg = paired_analysis_tables(df_ok)

print(f"Paired rows (all non-dense variants): {len(paired_all)}")
print(f"Paired rows (focused: window + best sparse per key): {len(paired)}")

display(agg.sort_values(["seq_len", "backend", "sparse_tag", "lr"]))

In [ ]:
winners = select_winners(agg, val_ppl_tolerance=PROFILE_CTX.val_ppl_tolerance)
print("Candidate wins over dense:")
display(winners)

In [ ]:
plot_speedup_ratios(paired)

In [ ]:
plot_vram_and_quality(paired)

In [ ]:
plots_dir = Path(run_result.runs_root) / "plots_all_metrics"
metric_cols = plot_all_numeric_metrics(df_ok, out_dir=plots_dir, show_inline=True)
display(metric_cols)